In [5]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import io
import sys
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

from typing import Tuple
from sklearn.base import RegressorMixin
from typing import Tuple, List
from sklearn.pipeline import Pipeline
from sklearn.base import RegressorMixin
from typing import Optional
from sklearn.base import ClassifierMixin

from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from typing import Tuple, List, Dict
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.pipeline import make_pipeline
import optuna.visualization as vis
from statsmodels.tsa.arima.model import ARIMA


from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import textwrap, pathlib, json, sys, inspect, os, types, importlib, pathlib, matplotlib, pandas as pd, numpy as np

#from run_ts_pipeline import run_ts_pipeline          # si lo guardas como módulo


In [6]:
import warnings, pathlib
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

def run_ts_pipeline(  # ← mismos parámetros que antes
        data_src, date_col, target_col,
        method="decompose", freq=None, steps=36,
        exog_cols=None, test_ratio=.2, random_state=123,
        plot=True, **method_kwargs):

    # ---------- carga y pre-procesado idénticos a la versión anterior ----------
    if isinstance(data_src, (str, pathlib.Path)):
        df = pd.read_csv(data_src) if str(data_src).lower().endswith(".csv") else pd.read_parquet(data_src)
    else:
        df = data_src.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)
    if freq: df = df.asfreq(freq)

    y = df[target_col]
    exog = df[exog_cols] if exog_cols else None
    split = int(len(df)*(1-test_ratio))
    y_train, y_test = y[:split], y[split:]
    exog_train = exog[:split] if exog is not None else None
    exog_test  = exog[split:] if exog is not None else None

    figures = []

    # --------------------- métodos ---------------------
    if method == "decompose":
        from statsmodels.tsa.seasonal import seasonal_decompose
        result = seasonal_decompose(y, model='additive', extrapolate_trend='freq')
        if plot:
            fig = result.plot(); fig.set_size_inches(8,6); figures.append(fig)
        return {"model": result, "fig": figures}

    elif method == "recursive_rf":
        from skforecast.ForecasterRecursive import ForecasterRecursive
        from sklearn.ensemble import RandomForestRegressor
        forecaster = ForecasterRecursive(
            regressor=RandomForestRegressor(random_state=random_state, **method_kwargs),
            lags=method_kwargs.get("lags", 12))
        forecaster.fit(y=y_train, exog=exog_train)
        pred = forecaster.predict(steps=steps, exog=exog_test)
        if plot:
            fig, ax = plt.subplots(figsize=(8,3))
            y_train.plot(ax=ax); y_test.plot(ax=ax); pred.plot(ax=ax,label="pred")
            ax.legend(); figures.append(fig)
        print(f"MSE test: {mean_squared_error(y_test[:len(pred)], pred):.3f}")
        return {"model": forecaster, "pred": pred, "train": y_train,
                "test": y_test, "fig": figures}

    elif method in ("auto_arima", "sarima"):
        import pmdarima as pm
        model = pm.auto_arima(
            y_train, exogenous=exog_train,
            seasonal=(method=="sarima"), m=method_kwargs.get("m", 1 if method=="auto_arima" else 12),
            stepwise=True, suppress_warnings=True, random_state=random_state,
            **{k:v for k,v in method_kwargs.items() if k not in ("m",)})
        pred, conf = model.predict(steps, exogenous=exog_test, return_conf_int=True)
        pred = pd.Series(pred, index=y_test.index[:steps])
        if plot:
            fig, ax = plt.subplots(figsize=(8,3))
            y_train.plot(ax=ax); y_test.plot(ax=ax)
            pred.plot(ax=ax,color="red"); ax.fill_between(pred.index, conf[:,0], conf[:,1], alpha=.2)
            ax.legend(); figures.append(fig)
        print(f"MSE test: {mean_squared_error(y_test[:steps], pred):.3f}")
        return {"model": model, "pred": pred, "train": y_train,
                "test": y_test, "fig": figures}

    else:
        raise ValueError("Método no soportado")


In [12]:
# o simplemente copia la definición previa en una celda y ejecútala

# 1) Descomposición estacional de un CSV mensual
run_ts_pipeline(
    data_src   = "../data/Datos_agregados.csv",
    date_col   = "Quincena",
    target_col = "Tipo_Mantenimiento",
    method     = "decompose",
    freq       = "MS"            # Monthly Start
)

ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
# 2) Forecast 36 pasos con RandomForest recursivo
run_ts_pipeline(
    data_src   = "Datos_agregados.csv",
    date_col   = "fecha",
    target_col = "y",
    method     = "recursive_rf",
    steps      = 36,
    exog_cols  = ["temperatura_media", "presion"],   # opcional
    lags       = 24,        # se pasa vía **method_kwargs
    n_estimators = 300,     # idem
)

In [ ]:
# 3) Auto-ARIMA sin estacionalidad
run_ts_pipeline(
    df, "fecha", "y",
    method = "auto_arima",
    steps  = 24,
    seasonal = False
)

In [ ]:
# 4) SARIMA automático (m=12 meses)
run_ts_pipeline(
    df, "fecha", "y",
    method = "sarima",
    steps  = 24,
    m      = 12
)
